In [ ]:
# 오류로 인해 아래 함수들을 04파일에서 정의함.

In [ ]:
import json
import os
import base64
import numpy as np
import cv2
from PIL import Image
from io import BytesIO

def load_annotations(json_path):
    """JSON 정답지 파일을 불러옴."""
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f" 데이터 로드 완료: 총 {len(data)}개의 항목")
        return data
    except FileNotFoundError:
        print(f" 오류: {json_path} 파일을 찾을 수 없음.")
        return []

def encode_image_to_base64(image):
    """PIL 이미지를 Base64 문자열로 변환함."""
    buffered = BytesIO()
    image.save(buffered, format="JPEG")
    return base64.b64encode(buffered.getvalue()).decode('utf-8')

# ---------------------------------------------------------
# [실험 A] 기존 VCD 방식 (가우시안 노이즈)
# ---------------------------------------------------------
def get_vcd_images(image_path, noise_step=500):
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"이미지가 없음: {image_path}")

    original_img = Image.open(image_path).convert('RGB')
    img_array = np.array(original_img, dtype=np.float32)
    
    std = noise_step / 10.0
    noise = np.random.normal(0, std, img_array.shape)
    
    noisy_img_array = np.clip(img_array + noise, 0, 255).astype(np.uint8)
    noisy_img = Image.fromarray(noisy_img_array)
    
    return original_img, noisy_img 

# ---------------------------------------------------------
# [실험 B] 제안 방식 B-VCD (푸아송-가우시안 + 모션 블러)
# ---------------------------------------------------------
def get_b_vcd_images(image_path, max_blur=30, brightness_factor=0.3, read_noise_std=5.0):
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"이미지가 없음: {image_path}")

    original_img = Image.open(image_path).convert('RGB')
    img_cv = np.array(original_img, dtype=np.float32)

    # 1. Stochastic Motion Blur
    l = np.random.randint(10, max_blur)
    theta = np.random.uniform(0, 180)
    M = cv2.getRotationMatrix2D((l / 2.0, l / 2.0), theta, 1)
    kernel = np.zeros((l, l))
    kernel[int((l - 1) / 2), :] = np.ones(l)
    kernel = cv2.warpAffine(kernel, M, (l, l))
    kernel = kernel / np.sum(kernel)
    blurred_cv = cv2.filter2D(img_cv, -1, kernel)

    # 2. Illumination Attenuation
    darkened_cv = blurred_cv * brightness_factor
    darkened_cv = np.nan_to_num(darkened_cv, nan=0.0)
    darkened_cv = np.clip(darkened_cv, 0.0, 255.0)
    
    # 3. Poisson-Gaussian Noise
    photon_scale = 10.0
    noisy_poisson = np.random.poisson(darkened_cv * photon_scale) / photon_scale
    read_noise = np.random.normal(0, read_noise_std, noisy_poisson.shape)
    
    final_distorted_cv = np.clip(noisy_poisson + read_noise, 0, 255).astype(np.uint8)
    domain_distorted_img = Image.fromarray(final_distorted_cv)
    
    return original_img, domain_distorted_img